In [19]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent
from pydantic import BaseModel, Field
import os
import asyncio
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [11]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [12]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [13]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Josh
"""

In [14]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [15]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [16]:
review.is_professional

False

# Guardrails

In [20]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

In [21]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [ ]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

In [25]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model='gpt-5.4-mini')
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Howdy — here’s a clean cold sales email draft:

**Subject:** Quick idea for [Company Name]

Hi [First Name],

Hope you’re doin’ well. I’m reachin’ out because we help teams like yours save time, cut down on busywork, and get better results without adding more tools to the pile.

At ComplAI, we work with companies that want to move faster and keep things simple. I thought it might be worth a quick chat to see if there’s a fit.

If you’re open to it, I’d be glad to share a few ideas tailored to [Company Name].

Best,  
[Your Name]  
ComplAI

If you want, I can make it sharper, more persuasive, or tailored to a specific industry.


In [26]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent
